# Python-1, Лекция 7

Лектор: Хайбулин Даниэль

Подготовил материал: Лущ Иван

Итак, сегодня мы поговорим про **исключения** и **контекстные менеджеры**.

### Исключения

Обработка ошибок — одна из наиболее дискуссионных тем в программировании. Можно сказать, что вокруг подходов к организации работы с ошибками во время выполнения программы формируются "религии" среди программистов.

Давайте рассмотрим ситуацию, в которой будет ошибка из-за попытки открытия несуществующего файла:

In [ ]:
with open("non_existent_file.txt", "r") as f:
    ...

Ошибки обрабатывают по-разному: где-то функция возвращает специальный код, где-то — специальное значение-результат, а где-то, как в `Python`, работает механизм **исключений**. Выбор влияет и на стиль кода, и на то, насколько легко его потом сопровождать.

Рассмотрим дополнительные примеры возникновений исключений:

In [ ]:
1 / 0

In [ ]:
int("not a number")

In [ ]:
print(not_defined_variable)

In [ ]:
[0] * 1_000_000_000_000_000_000

В `Python` для перехвата исключений используется конструкция `try...except`:

In [ ]:
try:
    ...  # код, который может вызвать исключение
except Exception:
    ...  # обработка исключения

Перехват исключений позволяет:

- Не прерывать работу программы при возникновении ошибки.
- Корректно реагировать на разные типы ошибок.
- Выдавать понятные сообщения, делать повторные попытки и т.д.

К примеру:

In [ ]:
try:
    x = 1 / 0
except ZeroDivisionError:
    print("Нельзя делить на ноль!")

Можно обрабатывать несколько типов исключений:

In [ ]:
for s in ("5", "0", "не число"):
    print(f"Ввод: {s!r}")
    try:
        n = int(s)
        print("  результат:", 10 / n)
    except ValueError:
        print("  Это не число!")
    except ZeroDivisionError:
        print("  Вы ввели ноль, деление невозможно.")

Можно перехватывать сразу несколько исключений, указав их в круглых скобках в виде кортежа (`tuple`) в блоке `except`:

In [ ]:
import typing as tp


def add_numbers(a: tp.Any, b: tp.Any) -> float | None:
    try:
        return float(a) + float(b)
    except (ValueError, TypeError) as err:
        print("Ошибка значения или типа:", err)
        return None


print(add_numbers("10", "5.5"))
print(add_numbers("abc", "3"))
print(add_numbers([1, 2], "3"))

Иногда бывает удобно перехватить сразу все «обычные» исключения (унаследованные от `Exception`), чтобы вывести ошибку пользователю (например, в пользовательском интерфейсе или логе), не прерывая всю программу аварийно:

In [ ]:
import tempfile
from pathlib import Path

# готовим файлы во временном каталоге, чтобы не сорить рядом с ноутбуком
tmp = Path(tempfile.mkdtemp())
(tmp / "positive.txt").write_text("5")
(tmp / "zero.txt").write_text("0")
(tmp / "text.txt").write_text("text")


def divide_by_file(path: Path) -> None:
    try:
        num = int(path.read_text().strip())
        result = 100 / num
    except Exception as exc:
        print(f"{path.name}: что-то пошло не так: {exc}")
        return

    print(f"{path.name}: 100 / {num} = {result}")


for name in ("positive.txt", "zero.txt", "text.txt", "non_existent.txt"):
    divide_by_file(tmp / name)

В конструкции `try...except` можно также использовать дополнительные блоки `else` и `finally`, чтобы более гибко управлять поведением программы в зависимости от наличия или отсутствия ошибок.


- Блок `else` срабатывает только если в блоке `try` не возникло исключения.
- Блок `finally` выполняется в любом случае: была ошибка или нет (например, для освобождения ресурсов).


In [ ]:
(tmp / "data.txt").write_text("I hate exceptions")


def open_and_read(path: Path) -> None:
    file = None
    try:
        file = open(path)
    except FileNotFoundError:
        print(f"{path.name}: файл не найден.")
    else:
        # сработает, только если в try не было исключения
        print(f"{path.name}: файл успешно открыт!")
        print("  первые 10 символов:", file.read(10))
    finally:
        # а этот блок выполнится в любом случае
        if file is not None:
            file.close()
            print(f"{path.name}: файл закрыт.")
        else:
            print(f"{path.name}: закрывать нечего.")


for name in ("data.txt", "non_existent.txt"):
    open_and_read(tmp / name)

В стандартной библиотеке Python определено множество видов исключений, структурированных в виде иерархии. Ниже приведена полная иерархия стандартных исключений:

https://docs.python.org/3/library/exceptions.html#exception-hierarchy

```
BaseException
 ├── BaseExceptionGroup
 ├── GeneratorExit
 ├── KeyboardInterrupt
 ├── SystemExit
 └── Exception
      ├── ArithmeticError
      │    ├── FloatingPointError
      │    ├── OverflowError
      │    └── ZeroDivisionError
      ├── AssertionError
      ├── AttributeError
      ├── BufferError
      ├── EOFError
      ├── ExceptionGroup [BaseExceptionGroup]
      ├── ImportError
      │    └── ModuleNotFoundError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── MemoryError
      ├── NameError
      │    └── UnboundLocalError
      ├── OSError
      │    ├── BlockingIOError
      │    ├── ChildProcessError
      │    ├── ConnectionError
      │    │    ├── BrokenPipeError
      │    │    ├── ConnectionAbortedError
      │    │    ├── ConnectionRefusedError
      │    │    └── ConnectionResetError
      │    ├── FileExistsError
      │    ├── FileNotFoundError
      │    ├── InterruptedError
      │    ├── IsADirectoryError
      │    ├── NotADirectoryError
      │    ├── PermissionError
      │    ├── ProcessLookupError
      │    └── TimeoutError
      ├── ReferenceError
      ├── RuntimeError
      │    ├── NotImplementedError
      │    ├── PythonFinalizationError
      │    └── RecursionError
      ├── StopAsyncIteration
      ├── StopIteration
      ├── SyntaxError
      │    └── IndentationError
      │         └── TabError
      ├── SystemError
      ├── TypeError
      ├── ValueError
      │    └── UnicodeError
      │         ├── UnicodeDecodeError
      │         ├── UnicodeEncodeError
      │         └── UnicodeTranslateError
      └── Warning
           ├── BytesWarning
           ├── DeprecationWarning
           ├── EncodingWarning
           ├── FutureWarning
           ├── ImportWarning
           ├── PendingDeprecationWarning
           ├── ResourceWarning
           ├── RuntimeWarning
           ├── SyntaxWarning
           ├── UnicodeWarning
           └── UserWarning
```


`BaseException` — "корень" всей иерархии; все остальные исключения наследуются от этого класса.
Исключения делятся на две больших ветви:

- Исключения, наследуемые от `BaseException` (**но НЕ от `Exception`**), например:
    - `KeyboardInterrupt` — прерывание с клавиатуры (`Ctrl+C`)
    
    - `SystemExit` — выход из интерпретатора
    
    - `GeneratorExit` — завершение генератора (узнаем чуть позже, что такое генераторы и что это за исключение)
    
    Эти обычно не стоит перехватывать в `except Exception`, иначе вы блокируете возможность корректного выхода из программы.

- Все остальные (от `Exception`) — это исключения, появляющиеся в повседневном коде, которые обычно мы и обрабатываем.


`try...except` без уточнения типа **перехватывает все исключения**, то есть любые экземпляры, наследуемые от `BaseException`, включая `SystemExit`, `KeyboardInterrupt`, `GeneratorExit` и даже пользовательские, которые напрямую наследуются от `BaseException`.

In [ ]:
try:
    raise KeyboardInterrupt("CTRL+C")
except (KeyboardInterrupt, ValueError) as exc:
    # несколько типов перечисляем кортежем; объединение через | здесь не работает
    print("Перехвачено!", exc)
finally:
    print("b")

Поэтому для обычных ошибок пишут `except Exception:`, а не голый `except:`. Иначе перехватится и `KeyboardInterrupt`, и программу нельзя будет остановить по `Ctrl+C`.

Важные группы `Exception`:


- `ArithmeticError`: ошибки арифметики — деление на ноль, переполнение, ошибки с плавающей точкой.
- `LookupError`: для неудачных обращений по индексу (`IndexError`) и ключу (`KeyError`).
- `OSError`: все ошибки, связанные с операционной системой — файлы, процессы, соединения.
- `RuntimeError`, `ValueError`, `TypeError` — "общие" ошибки, возникающие при неправильных входных данных, нарушениях логики и т.п.
- `ImportError`, `ModuleNotFoundError`: ошибки импорта модулей.
- `SyntaxError`, `IndentationError`, `TabError`: нарушения синтаксиса, проблемы с отступами в исходном коде.


![](1f11cc1d97095debe72a443bc576e2ed.jpg)

`IndentationError` — это разновидность `SyntaxError` и относится к исключениям уровня компиляции, возникает во время разбора (парсинга) кода `Python`, до его фактического выполнения. Поэтому перехватить `IndentationError` в обычном `try...except` вокруг кода с ошибкой невозможно — такой код даже не исполнится.

In [ ]:
def test():
print("Ошибка отступа!")

Однако можно обработать ситуацию, когда вы исполняете или компилируете строку с ошибочной программой во время выполнения. Например, с помощью функции `exec()`:

In [ ]:
code = """
def test():
print("Ошибка отступа!")
"""

try:
    # функция exec выполняет код, представленный в виде строки
    exec(code)
except IndentationError as e:
    print("IndentationError:", e)
except SyntaxError as e:
    print("Другая синтаксическая ошибка:", e)

Особняком в иерархии стоит `Warning`: его подклассы — не ошибки, а сообщения-предупреждения, выполнение программы они не прерывают. Управляют ими через модуль `warnings`; в этом курсе мы их подробно не разбираем.

С `Python 3.11` есть ещё **группы исключений** (`ExceptionGroup`) и конструкция `except*` — на случай, когда одновременно возникло несколько ошибок. Нужны они в основном в асинхронном коде, поэтому здесь мы их не разбираем.

Теперь, когда мы разобрались с перехватом исключений, рассмотрим, как инициировать их возникновение самостоятельно (так называемое "бросание" исключений):

In [ ]:
raise Exception("Исключение!")

Оператор `raise` требует, чтобы в качестве аргумента ему передавался объект, являющийся экземпляром класса, производного от `BaseException`. Передача в качестве аргумента объекта, не являющегося экземпляром класса, производного от `BaseException` (например, строки или числа), приведёт к возникновению исключения `TypeError`:

In [ ]:
raise 1

In [ ]:
# здесь все ок
a = ValueError("value error")

raise a

А ещё можно использовать `raise` внутри блока `except` без указания типа исключения — это позволяет повторно пробросить текущее исключение после выполнения необходимой дополнительной обработки, например, логирования или очистки ресурсов:

In [ ]:
try:
    try:
        1 / 0
    except ZeroDivisionError:
        print("Логируем ошибку и пробрасываем дальше")
        raise  # без аргументов: пробрасывает текущее исключение как есть
except ZeroDivisionError as err:
    print("Наружу вышло то же самое исключение:", type(err).__name__)

Ещё одной важной возможностью является использование конструкции `raise ... from ...` С её помощью можно явно указать причинно-следственную связь между двумя исключениями: если одно исключение возникает в процессе обработки другого, новое исключение может быть выброшено с ссылкой на исходное. Такой приём называется **цепочкой исключений** (`exception chaining`) и облегчает отладку и анализ ошибок, позволяя видеть, какое исключение послужило причиной нового.

In [ ]:
try:
    int("abc")
except ValueError as e:
    raise RuntimeError("Ошибка преобразования данных") from e

Цепочка исключений доступна не только в трейсбэке, но и прямо из кода — через два поля, и разница между ними принципиальна:

* `__cause__` заполняется при `raise ... from ...`. Автор кода **явно заявил**: вот настоящая причина;
* `__context__` заполняется автоматически, если новое исключение возникло внутри блока `except`. Никто ничего не заявлял, связь получилась сама собой;
* `__suppress_context__` становится `True` при любом `raise ... from ...`, в том числе при `raise ... from None`, которым связь обрывают намеренно.

In [ ]:
from collections.abc import Callable


def catch(fn: Callable[[], None]) -> BaseException:
    """Возвращает исключение, которое бросила функция."""
    try:
        fn()
    except BaseException as exc:
        return exc
    raise AssertionError("исключения не было")


def explicit() -> None:
    try:
        int("abc")
    except ValueError as e:
        raise RuntimeError("причина указана явно") from e


def implicit() -> None:
    try:
        int("abc")
    except ValueError:
        raise RuntimeError("причина получилась сама собой")


def suppressed() -> None:
    try:
        int("abc")
    except ValueError:
        raise RuntimeError("связь оборвана") from None


for fn in (explicit, implicit, suppressed):
    exc = catch(fn)
    print(
        f"{fn.__name__:10} "
        f"__cause__={type(exc.__cause__).__name__:9} "
        f"__context__={type(exc.__context__).__name__:9} "
        f"__suppress_context__={exc.__suppress_context__}"
    )

Отсюда правило. `raise ... from ...` пишут, когда причина известна и её стоит сохранить. `raise ... from None` — когда старое исключение только мешает читать лог. А `__context__`, появившийся без вашего участия, стоит читать как сигнал: возможно, в самом обработчике что-то пошло не так.

В некоторых случаях стандартных исключений Python бывает недостаточно для выражения специфических ошибок, характерных именно для вашей программы или библиотеки. В таких ситуациях рекомендуется определять собственные типы исключений:

In [ ]:
class MyCustomError(Exception):
    """Описание пользовательской ошибки"""

    ...


# Использование собственного исключения:
raise MyCustomError("Что-то пошло не так")

Рекомендуется давать пользовательским классам исключений имена, оканчивающиеся на `Error` ([pep8](https://peps.python.org/pep-0008/#exception-names)).

In [ ]:
class MySubCustomError(MyCustomError):
    def __str__(self):
        # self.args - это аргументы, которые мы передали в __init__
        return f"Some information of my sub custom error: {self.args}"


try:
    raise MySubCustomError(1)
except MyCustomError as e:
    print(f"Что-то не так: {e=}")
    raise

Своё исключение полезно не только именем — оно может нести данные о том, что именно случилось. Для этого достаточно обычного `__init__`, а вызов `super().__init__(...)` задаёт текст сообщения:

In [ ]:
class ConfigLineError(Exception):
    def __init__(self, line: int, reason: str) -> None:
        super().__init__(f"строка {line}: {reason}")
        self.line = line
        self.reason = reason


try:
    raise ConfigLineError(12, "ожидалось 'ключ = значение'")
except ConfigLineError as err:
    print(err)  # сообщение, которое собрал super().__init__
    print(err.line, err.reason)  # а это - разобранные поля

В `Python` есть модуль `traceback` — он даёт текстовое представление трассировки стека (`stack trace`). Познакомиться с ним, как и с атрибутом `__traceback__`, предлагается на семинарских занятиях.

### Контекстные менеджеры

Не следует пугаться термина "контекстный менеджер" — вы уже сталкивались с их использованием:

In [ ]:
with open("file.txt", "w") as f:
    ...

Контекстные менеджеры в Python обеспечивают автоматическое выполнение определённых действий при входе в блок кода и выходе из него. Cпособ их использования — конструкция `with`.

В приведённом выше примере при входе в блок осуществляется открытие файла для чтения, а при выходе из блока файл автоматически закрывается.

Функция open возвращает объект, реализующий протокол контекстного менеджера, то есть содержащий специальные методы:
- `__enter__` в данном случае возвращает сам объект файла (возвращает `self`)
- `__exit__` обеспечивает автоматический вызов метода `close` для закрытия файла при выходе из блока управления контекстом.

Перейдем к определению. Контекстные менеджеры --- это объекты, реализующие протокол контекстного менеджера (методы `__enter__` и `__exit__`).

Рассмотрим вопрос о назначении контекстных менеджеров. Для чего они необходимы? Контекстные менеджеры гарантируют нам, что определённое действие будет выполнено вне зависимости от того, возникло ли в процессе работы программы исключение или нет:


Без использования контекстных менеджеров обеспечение корректного освобождения ресурсов требует явной обработки, что может привести к ошибкам или утечкам ресурсов. Рассмотрим следующий пример:

In [ ]:
def process_file(f):
    # Какая-то потенциально опасная операция с файлом
    f.write("Начинается запись данных...\n")
    raise RuntimeError("Возникла критическая ошибка при обработке файла!")


f = open("file.txt", "w")
try:
    process_file(f)
except Exception as exc:
    print(f"Произошло исключение: {exc}")
finally:
    print("Закрываем файл.")
    try:
        f.close()
        print("Файл успешно закрыт.")
    except Exception as close_exc:
        print(f"Ошибка при закрытии файла: {close_exc}")

В данном случае необходимо явно вызывать метод `close()` для закрытия файла, даже если в процессе работы возникает исключение.

Контекстные менеджеры предоставляют удобный и безопасный механизм для инициализации и гарантированного завершения работы с ресурсом (финализации "контекста"), устраняя необходимость ручного контроля освобождения ресурсов и снижая риск возникновения ошибок:

In [ ]:
with open("file.txt", "w") as f:
    process_file(f)

Такой подход — «захватил ресурс при входе в блок, освободил при выходе» — называют **RAII** (Resource Acquisition Is Initialization). Полнее вы увидите его в `C++`, где освобождение ресурса жёстко привязано ко времени жизни объекта.

Приведу несколько примеров использования контекстных менеджеров из стандартной библиотеки `Python` (в основном из модуля `contextlib`). Здесь не будет примеров работы с соединениями к базам данных, так как предполагаю, что большинство с ними пока не сталкивались. Тем не менее, при необходимости подобные примеры легко найти — можете погуглить или попросить какую-нибудь нейросеть.

In [ ]:
import tempfile

# Создание временного файла
with tempfile.TemporaryFile(mode="w+t") as temp:
    temp.write("Temporary content")
    temp.seek(0)
    print(temp.read())

# Файл автоматически удаляется

In [ ]:
import contextlib

with contextlib.suppress(FileNotFoundError):
    with open("no_such_file.txt") as f:
        print(f.read())

# Исключение FileNotFoundError будет проигнорировано

Давайте рассмотрим пример собственного контекстного менеджера, который будет измерять время выполнения кода внутри блока `with`. Для этого можно реализовать класс с методами `__enter__` и `__exit__`:

In [ ]:
import time
from types import TracebackType


class Timer:
    def __enter__(self) -> "Timer":
        self._start_time: float = time.time()
        print("Выполнение началось.")
        return self

    def __exit__(
        self,
        exctype: type[BaseException] | None,
        excinst: BaseException | None,
        exctb: TracebackType | None,
    ) -> bool | None:
        self._end_time: float = time.time()
        duration: float = self._end_time - self._start_time
        print(f"Выполнение завершено. Прошло времени: {duration:.4f} секунд.")

Давайте разберем аргументы `__exit__`:

- `exctype: type[BaseException] | None` - это любой тип исключения, а `| None` допускает ситуацию без исключения.

- `excinst: BaseException | None` - экземпляр возникшего исключения, если исключения не было, будет передано `None`.

- `exctb: TracebackType | None` - трейсбэк (объект, описывающий стек вызовов в момент возникновения исключения), если исключение не возникло, тут будет `None`.

Возвращаемое значение `bool | None`:
- если метод `__exit__` возвращает `True`, то возникшее исключение считается обработанным и не будет передано дальше.
- если возвращается `False` или `None`, исключение будет проброшено дальше.

In [ ]:
with Timer():
    time.sleep(2)

Давайте сделаем еще один пример, в котором поработаем с аргументами `__exit__`:

In [ ]:
import traceback
from types import TracebackType


class ExceptionLogger:
    def __init__(self, raise_exception: bool = False):
        self._raise_exception = raise_exception

    def __enter__(self) -> "ExceptionLogger":
        print("Вход в контекстный менеджер")
        return self

    def __exit__(
        self,
        exctype: type[BaseException] | None,
        excinst: BaseException | None,
        exctb: TracebackType | None,
    ) -> bool | None:
        if exctype is not None:
            print("Обнаружено исключение:")
            print(f"Тип: {exctype.__name__}")
            print(f"Сообщение: {excinst}")
            print("Traceback:")
            traceback.print_tb(exctb)
        else:
            print("Выход из контекстного менеджера без исключений.")
        return not self._raise_exception

In [ ]:
with ExceptionLogger():
    print("До исключения")
    1 / 0
    print("Этот код не выполнится")

In [ ]:
with ExceptionLogger(raise_exception=True):
    print("До исключения")
    1 / 0
    print("Этот код не выполнится")

In [ ]:
with ExceptionLogger(raise_exception=True):
    print("Этот код выполнится")

Существует более удобный способ создания собственных контекстных менеджеров — с помощью декоратора `@contextmanager` из модуля `contextlib`. Однако для его понимания необходимо предварительно познакомиться с такими концепциями, как генераторы и декораторы. Эти темы будут подробно рассмотрены на последующих лекциях, и тогда мы разберём альтернативный, более лаконичный подход к написанию контекстных менеджеров.